# Add: Embeddings, and Linear Output Layer

In [52]:
data = [
  "She composes songs and practices piano daily.",
  "He reads books and explores the nearby caves.",
  "He reads novel and climbs mountains every weekend.",
  "She composes songs and writes novels.",
  "He reads newspaper and solves complex puzzles.",
  "She composes music and organizes exhibitions regularly.",
  "He reads books and builds small wooden models.",
  "He reads books and participates in local science fairs.",
  "She composes songs and curates art projects.",
  "She composes tunes and designs jewelry for her friends.",
  "He reads everyday and documents wildlife photography trips.",
  "She composes harmonies and experiments with digital music.",
  "He reads novel and trains for local marathons.",
  "She composes soundtracks and collaborates with creative filmmakers.",
  "He reads newspaper and studies navigation using maps and stars.",
  "She composes rhythms and teaches music."
]

In [53]:
VOCAB_SIZE = 70
CONTEXT_LEN = 6
EMB_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [54]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

In [55]:
_ = torch.manual_seed(123)

In [56]:
def seprate_dot(S):
    s = re.sub(r'([.,!?])', r' \1 ', S)
    return s.strip()

text_data = [seprate_dot(s) for s in data]

In [57]:
example_data = text_data[0]
print(example_data)

She composes songs and practices piano daily .


In [58]:
vocab = set()

for text in text_data:
    for word in text.split():
        vocab.add(word)

vocab = sorted(vocab)
print(len(vocab))

70


In [59]:
for i, word in enumerate(vocab):
    print(i, word)

0 .
1 He
2 She
3 and
4 art
5 books
6 builds
7 caves
8 climbs
9 collaborates
10 complex
11 composes
12 creative
13 curates
14 daily
15 designs
16 digital
17 documents
18 every
19 everyday
20 exhibitions
21 experiments
22 explores
23 fairs
24 filmmakers
25 for
26 friends
27 harmonies
28 her
29 in
30 jewelry
31 local
32 maps
33 marathons
34 models
35 mountains
36 music
37 navigation
38 nearby
39 newspaper
40 novel
41 novels
42 organizes
43 participates
44 photography
45 piano
46 practices
47 projects
48 puzzles
49 reads
50 regularly
51 rhythms
52 science
53 small
54 solves
55 songs
56 soundtracks
57 stars
58 studies
59 teaches
60 the
61 trains
62 trips
63 tunes
64 using
65 weekend
66 wildlife
67 with
68 wooden
69 writes


In [60]:
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

In [61]:
def text_to_token(text,word_to_idx):
    tokens = text.split()
    token_ids = [word_to_idx[token] for token in tokens]
    return token_ids

def token_to_text(token_ids, idx_to_word):
    words = [idx_to_word[token_id] for token_id in token_ids]
    text = ' '.join(words)
    return text

In [62]:
print(text_to_token(example_data, word_to_idx))
print(token_to_text(text_to_token(example_data, word_to_idx), idx_to_word))

[2, 11, 55, 3, 46, 45, 14, 0]
She composes songs and practices piano daily .


In [63]:
text_to_token(example_data, word_to_idx)[: CONTEXT_LEN + 1]

[2, 11, 55, 3, 46, 45, 14]

In [64]:
text_to_token(example_data, word_to_idx)[: CONTEXT_LEN ]

[2, 11, 55, 3, 46, 45]

In [65]:
text_to_token(example_data, word_to_idx)[: CONTEXT_LEN - 1]

[2, 11, 55, 3, 46]

In [66]:
text_to_token(example_data, word_to_idx)[1: CONTEXT_LEN + 1]

[11, 55, 3, 46, 45, 14]

In [67]:
text_to_token(example_data, word_to_idx)[1: CONTEXT_LEN ]

[11, 55, 3, 46, 45]

In [68]:
class LLMDataset(Dataset):
    def __init__(self, text_data, word_to_idx, context_len):
        self.text_data = text_data
        self.word_to_idx = word_to_idx
        self.context_len = context_len
    
    def __len__(self):
         return len(self.text_data)

    def __getitem__(self, idx):
        tokens = torch.tensor(text_to_token(self.text_data[idx], self.word_to_idx)[: self.context_len + 1])
        x = tokens[:-1]
        y = tokens[1:]
        return x,y
    
train_dataset = LLMDataset(text_data, word_to_idx, CONTEXT_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(len(train_loader))


4


In [69]:
example_input_output = next(iter(train_loader))

In [70]:
print(example_input_output)

[tensor([[ 2, 11, 55,  3, 46, 45],
        [ 1, 49,  5,  3, 22, 60],
        [ 1, 49, 40,  3,  8, 35],
        [ 2, 11, 55,  3, 69, 41]]), tensor([[11, 55,  3, 46, 45, 14],
        [49,  5,  3, 22, 60, 38],
        [49, 40,  3,  8, 35, 18],
        [11, 55,  3, 69, 41,  0]])]


In [71]:
example_input_output[0][0]

tensor([ 2, 11, 55,  3, 46, 45])

In [72]:
example_input_output[1][0]

tensor([11, 55,  3, 46, 45, 14])

In [73]:
class GPTModel(nn.Module):


    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMB_DIM)
        self.output_layer = nn.Linear(EMB_DIM, VOCAB_SIZE, bias=False)
    
    def forward(self, in_idx):
       _, seq_len = in_idx.shape
       tok_embeds = self.tok_emb(in_idx)
       pos_embeds = self.pos_emb(torch.arange(seq_len))
       x = tok_embeds + pos_embeds
       logits = self.output_layer(x)
       return logits

In [74]:
model = GPTModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

In [75]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total parameter to trained: {trainable_params:,}")

total parameter to trained: 3,504


In [76]:
total_params = sum(p.numel() for p in model.parameters())
print(f"total parameter in the model: {total_params:,}")

total parameter in the model: 3,504


In [77]:
from collections import defaultdict
param_count = defaultdict(int)

for name, param in model.named_parameters():
    top_level = name.split('.')[0]
    param_count[top_level] += param.numel()

for k, v in param_count.items():
    print(f"{k}: {v:,}")    


tok_emb: 1,680
pos_emb: 144
output_layer: 1,680


In [78]:
def train (dataloader, model,loss_fn, optimizer):
    model.train()
    
    for batch_idx, (x, y) in enumerate(dataloader):
        logits = model(x)
        loss = loss_fn(logits.flatten(0, 1), y.flatten())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"batch {batch_idx + 1} loss: {loss:>4f}")
    

In [79]:
for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
    print("\n")
    print("-------------------------------")
    print("DONE")


Epoch 1
-------------------------------
batch 1 loss: 4.721181
batch 2 loss: 4.726192
batch 3 loss: 4.490270
batch 4 loss: 4.470550


-------------------------------
DONE
Epoch 2
-------------------------------
batch 1 loss: 4.403049
batch 2 loss: 4.464782
batch 3 loss: 4.271320
batch 4 loss: 4.256333


-------------------------------
DONE
Epoch 3
-------------------------------
batch 1 loss: 4.152456
batch 2 loss: 4.233850
batch 3 loss: 4.063265
batch 4 loss: 4.044014


-------------------------------
DONE
Epoch 4
-------------------------------
batch 1 loss: 3.918307
batch 2 loss: 4.015189
batch 3 loss: 3.863507
batch 4 loss: 3.837523


-------------------------------
DONE
Epoch 5
-------------------------------
batch 1 loss: 3.695820
batch 2 loss: 3.806128
batch 3 loss: 3.671575
batch 4 loss: 3.637664


-------------------------------
DONE
Epoch 6
-------------------------------
batch 1 loss: 3.483142
batch 2 loss: 3.605291
batch 3 loss: 3.487070
batch 4 loss: 3.444558


-----------

In [80]:
_= model.eval()

In [81]:
def generate(start_text):
    token_ids = text_to_token(start_text, word_to_idx)
    num_new_tokens = CONTEXT_LEN - len(token_ids)
    idx = torch.tensor(token_ids).unsqueeze(0)
    for _ in range(num_new_tokens):
        idx_cond = idx[:, -CONTEXT_LEN :]
        logits = model(idx_cond)
        idx_next = torch.argmax(logits[:, -1, :], dim=-1)
        idx = torch.cat((idx, idx_next.unsqueeze(0)), dim=1)

    idx = idx.view(-1).tolist()
    text = token_to_text(idx, idx_to_word)
    return text    

In [82]:
text = "He reads"
generated_text = generate(text)
print(generated_text)

He reads books and experiments with


In [83]:
text = "She composes"
generated_text = generate(text)
print(generated_text)

She composes songs and experiments with
